# Crosswind 1-cosine gust 6DOF response example

このNotebookでは、`SampleGlider.stab`の線形空力モデルを使い、横方向の1-cosine gustに対する6自由度応答を計算する。

横風突風の符号規約は次で統一する。

$$
U_{ds}>0
\quad:\quad
\text{航空機の右側から左側へ吹く横風突風}
$$

公開時刻歴では、

- `gust_velocity_y > 0`：右から吹く横風突風
- `airmass_velocity_y < 0`：機体軸$+y_b$右方に対して左向きの大気速度
- `beta_g > 0`：右からの突風による正の突風横滑り角

となる。

ここで使う質量・慣性モーメントは`SampleGlider`用の数値例であり、G103Aの実機MassProp値ではない。


## 1. importとリポジトリ位置


In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

start_dir = Path.cwd().resolve()
repo_root = next(
    (
        candidate
        for candidate in [start_dir, *start_dir.parents]
        if (candidate / "src" / "RollRudderGain.py").exists()
    ),
    None,
)

if repo_root is None:
    raise FileNotFoundError(
        "Could not find the repository root containing "
        "src/RollRudderGain.py."
    )

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.RollRudderGain import (  # noqa: E402
    calculate_crosswind_gust_response_indices,
    one_cosine_gust_velocity,
    plot_6dof_history,
    simulate_6dof_crosswind_gust_from_stab,
    write_6dof_history_csv,
)
from src.TrimTurnSolver import read_vspaero_stab  # noqa: E402

print("repo_root:", repo_root)


## 2. 入力モデルと解析条件

`Uds=3.0`は右から吹くピーク突風速度を表す。

`H=9.144 m`は30 ftの突風勾配距離である。


In [ ]:
stab_path = (
    repo_root
    / "examples"
    / "models"
    / "SampleGlider"
    / "SampleGlider.stab"
)
output_dir = (
    repo_root
    / "examples"
    / "notebooks"
    / "results"
    / "crosswind_gust_response"
)
output_dir.mkdir(parents=True, exist_ok=True)

if not stab_path.exists():
    raise FileNotFoundError(stab_path)

stab = read_vspaero_stab(stab_path)

mass = 100.0
inertia = {
    "Ixx": 1000.0,
    "Iyy": 75.0,
    "Izz": 1100.0,
    "Ixz": 0.0,
}

Uds = 3.0
H = 9.144
gust_start_time = 0.0
gust_end_time = gust_start_time + 2.0 * H / stab.V0
t_final = gust_end_time + 1.2

display(
    pd.Series(
        {
            "stab_path": str(stab_path),
            "mass": mass,
            "Ixx": inertia["Ixx"],
            "Iyy": inertia["Iyy"],
            "Izz": inertia["Izz"],
            "Ixz": inertia["Ixz"],
            "Vinf": stab.V0,
            "Bref": stab.Bref,
            "rho": stab.rho0,
            "Uds_from_right": Uds,
            "H": H,
            "gust_end_time": gust_end_time,
            "t_final": t_final,
        },
        name="value",
    )
)


## 3. 1-cosine gust波形

突風通過後に速度が0へ戻ることも含めて確認する。


In [ ]:
time = np.linspace(
    0.0,
    1.25 * gust_end_time,
    300,
)
gust_velocity_y = np.array(
    [
        one_cosine_gust_velocity(
            t,
            Uds=Uds,
            H=H,
            V=stab.V0,
            start_time=gust_start_time,
        )
        for t in time
    ]
)

fig, ax = plt.subplots(
    figsize=(7, 4),
    constrained_layout=True,
)
ax.plot(time, gust_velocity_y)
ax.axvline(
    gust_end_time,
    linestyle="--",
    label="gust end",
)
ax.set_xlabel("time [s]")
ax.set_ylabel(
    "gust speed from right [m/s]"
)
ax.grid(True)
ax.legend()
plt.show()


## 4. 横風突風6自由度応答の計算

突風速度は空力相対風へだけ加え、機体の剛体速度状態そのものへ直接加えない。


In [ ]:
history = simulate_6dof_crosswind_gust_from_stab(
    stab_path,
    mass=mass,
    Ixx=inertia["Ixx"],
    Iyy=inertia["Iyy"],
    Izz=inertia["Izz"],
    Ixz=inertia["Ixz"],
    Uds=Uds,
    H=H,
    t_final=t_final,
    delta_a=0.0,
    delta_r=0.0,
    delta_e=None,
    trim_elevator=True,
    trim_thrust=False,
    phi0=0.0,
    theta0=None,
    psi0=0.0,
    gust_start_time=gust_start_time,
    max_step=0.02,
)

sign_conventions = set(
    history[
        "crosswind_sign_convention"
    ].dropna().astype(str)
)
assert sign_conventions == {
    "positive_from_right"
}
assert history["gust_velocity_y"].max() > 0.0
assert history["airmass_velocity_y"].min() < 0.0
assert history["beta_g"].max() > 0.0

display(
    history[
        [
            "time",
            "gust_velocity_y",
            "airmass_velocity_y",
            "beta_g",
            "beta",
            "p",
            "r",
            "phi",
        ]
    ].head()
)
print("samples:", len(history))


## 5. 横風突風応答指標

最大絶対バンク角変化、横滑り角、無次元ロール率、無次元ヨーレートなどを確認する。


In [ ]:
gust_response_indices = (
    calculate_crosswind_gust_response_indices(
        history
    )
)

display(
    pd.Series(
        {
            "success": gust_response_indices[
                "crosswind_gust_success"
            ],
            "Uds_from_right": gust_response_indices[
                "crosswind_gust_Uds"
            ],
            "H": gust_response_indices[
                "crosswind_gust_H"
            ],
            "max_abs_phi_delta_deg": math.degrees(
                gust_response_indices[
                    "crosswind_gust_max_abs_phi_delta"
                ]
            ),
            "max_abs_phi_delta_time": (
                gust_response_indices[
                    "crosswind_gust_max_abs_phi_delta_time"
                ]
            ),
            "max_abs_beta_deg": math.degrees(
                gust_response_indices[
                    "crosswind_gust_max_abs_beta"
                ]
            ),
            "max_abs_phat": gust_response_indices[
                "crosswind_gust_max_abs_phat"
            ],
            "max_abs_rhat": gust_response_indices[
                "crosswind_gust_max_abs_rhat"
            ],
            "bank_index_abs_per_beta_g": (
                gust_response_indices[
                    "crosswind_gust_bank_index_abs_per_beta_g"
                ]
            ),
            "message": gust_response_indices[
                "crosswind_gust_message"
            ],
        },
        name="value",
    )
)


## 6. 時刻歴の保存と表示


In [ ]:
csv_path = write_6dof_history_csv(
    history,
    output_dir
    / "crosswind_gust_6dof_history.csv",
)

plot_path = (
    output_dir
    / "crosswind_gust_6dof_history.png"
)
plot_6dof_history(
    history,
    plot_path=plot_path,
    show=True,
    degrees=True,
)

print("csv_path:", csv_path)
print("plot_path:", plot_path)


## 7. 確認事項

- `crosswind_sign_convention`が`positive_from_right`
- `Uds>0`のとき`gust_velocity_y>0`
- `Uds>0`のとき`airmass_velocity_y<0`
- `Uds>0`のとき`beta_g>0`
- 突風通過後に`gust_velocity_y=0`
- 質量・慣性モーメントと`.stab`が同じ単位系
- 線形微係数の適用範囲を超える大きな横滑り角になっていないか
